In [4]:
# Print the first 5 lines of the file exactly as they are
with open("diabetic_data.csv") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i == 4:
            break

encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,

In [5]:
import pandas as pd

# Load without telling pandas anything about missing values yet
df = pd.read_csv("diabetic_data.csv")

# Rows and columns, and the memory it uses
print(df.shape)
print(round(df.memory_usage(deep=True).sum() / 1e6, 1), "MB")

# Column types
print(df.dtypes)

(101766, 50)
202.2 MB
encounter_id                 int64
patient_nbr                  int64
race                        object
gender                      object
age                         object
weight                      object
admission_type_id            int64
discharge_disposition_id     int64
admission_source_id          int64
time_in_hospital             int64
payer_code                  object
medical_specialty           object
num_lab_procedures           int64
num_procedures               int64
num_medications              int64
number_outpatient            int64
number_emergency             int64
number_inpatient             int64
diag_1                      object
diag_2                      object
diag_3                      object
number_diagnoses             int64
max_glu_serum               object
A1Cresult                   object
metformin                   object
repaglinide                 object
nateglinide                 object
chlorpropamide              objec

In [6]:
# Percent of values that are "?" in each column (only show columns that have some)
q = (df == "?").mean().mul(100).round(1)
print(q[q > 0].sort_values(ascending=False))

weight               96.9
medical_specialty    49.1
payer_code           39.6
race                  2.2
diag_3                1.4
diag_2                0.4
dtype: float64


In [7]:
# Does "None" show as text, or did pandas turn it into missing (NaN)?
print(df["max_glu_serum"].value_counts(dropna=False))
print(df["A1Cresult"].value_counts(dropna=False))

# The label: counts and shares
print(df["readmitted"].value_counts())
print(df["readmitted"].value_counts(normalize=True).round(3))

max_glu_serum
NaN     96420
Norm     2597
>200     1485
>300     1264
Name: count, dtype: int64
A1Cresult
NaN     84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64
readmitted
NO     0.539
>30    0.349
<30    0.112
Name: proportion, dtype: float64


In [8]:
# Rows (hospital stays) vs different patients
print("Rows:", len(df), "| Unique patients:", df["patient_nbr"].nunique())

# How many patients came back more than once, and the most stays by one patient
counts = df["patient_nbr"].value_counts()
print("Patients with more than 1 stay:", (counts > 1).sum())
print("Most stays by one patient:", counts.max())

# Patients who died or went to hospice can't be readmitted (codes 11, 13, 14, 19, 20, 21)
died_or_hospice = [11, 13, 14, 19, 20, 21]
print("Died or hospice rows:", df["discharge_disposition_id"].isin(died_or_hospice).sum())

Rows: 101766 | Unique patients: 71518
Patients with more than 1 stay: 16773
Most stays by one patient: 40
Died or hospice rows: 2423


In [9]:
# Discharge codes for died (11, 19, 20, 21) and hospice (13, 14)
died_or_hospice = [11, 13, 14, 19, 20, 21]
mask = df["discharge_disposition_id"].isin(died_or_hospice)

# Their readmission labels
print(df.loc[mask, "readmitted"].value_counts())

readmitted
NO     2337
<30      43
>30      43
Name: count, dtype: int64


In [10]:
# Remove died or hospice patients
df = df[~mask].copy()
print("Rows left:", len(df))

# Binary label: 1 = readmitted within 30 days, 0 = everything else
df["target"] = (df["readmitted"] == "<30").astype(int)
print(df["target"].value_counts())
print(df["target"].value_counts(normalize=True).round(3))

Rows left: 99343
target
0    88029
1    11314
Name: count, dtype: int64
target
0    0.886
1    0.114
Name: proportion, dtype: float64


In [11]:
from sklearn.model_selection import GroupShuffleSplit

# First split: 70% of patients for training, 30% set aside
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(df, df["target"], groups=df["patient_nbr"]))
train_df = df.iloc[train_idx]
temp_df = df.iloc[temp_idx]

# Second split: divide the 30% in half, 15% validation and 15% test (still by patient)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(temp_df, temp_df["target"], groups=temp_df["patient_nbr"]))
val_df = temp_df.iloc[val_idx]
test_df = temp_df.iloc[test_idx]

In [12]:
# Rows, share of early readmissions, and number of patients in each part
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(part), round(part["target"].mean(), 3), part["patient_nbr"].nunique())

# No patient should appear in more than one part (these should all print 0)
print(len(set(train_df["patient_nbr"]) & set(val_df["patient_nbr"])))
print(len(set(train_df["patient_nbr"]) & set(test_df["patient_nbr"])))
print(len(set(val_df["patient_nbr"]) & set(test_df["patient_nbr"])))

train 69519 0.114 48993
val 14911 0.117 10498
test 14913 0.111 10499
0
0
0


In [13]:
# Columns that must not be features:
# the two ids, the original label (readmitted) and the new label (target)
drop_cols = ["encounter_id", "patient_nbr", "readmitted", "target"]

X_train = train_df.drop(columns=drop_cols)
X_val = val_df.drop(columns=drop_cols)
X_test = test_df.drop(columns=drop_cols)

# The label: 1 = readmitted within 30 days, 0 = otherwise
y_train = train_df["target"]
y_val = val_df["target"]
y_test = test_df["target"]

# Keep patient ids separately, needed later to cross-validate by patient
g_train = train_df["patient_nbr"]

print(X_train.shape, X_val.shape, X_test.shape)

(69519, 47) (14911, 47) (14913, 47)


In [14]:
# For each column in the training set:
#   unique = how many different values it has
#   top_share = share of the most common value (missing values counted too)
#   question = share of "?" values
info = pd.DataFrame({
    "unique": X_train.nunique(),
    "top_share": X_train.apply(lambda c: c.value_counts(normalize=True, dropna=False).iloc[0]).round(3),
    "question": (X_train == "?").mean().round(3),
})
pd.set_option("display.max_rows", 100)
print(info.sort_values("top_share", ascending=False))

                          unique  top_share  question
examide                        1      1.000     0.000
tolazamide                     3      1.000     0.000
troglitazone                   2      1.000     0.000
citoglipton                    1      1.000     0.000
miglitol                       3      1.000     0.000
glipizide-metformin            2      1.000     0.000
glimepiride-pioglitazone       1      1.000     0.000
metformin-rosiglitazone        2      1.000     0.000
metformin-pioglitazone         2      1.000     0.000
tolbutamide                    2      1.000     0.000
acetohexamide                  2      1.000     0.000
chlorpropamide                 4      0.999     0.000
acarbose                       3      0.997     0.000
glyburide-metformin            4      0.993     0.000
nateglinide                    4      0.993     0.000
repaglinide                    4      0.984     0.000
weight                        10      0.969     0.969
max_glu_serum               

In [15]:
# Columns where one value covers 99% or more of the training patients (almost no information)
near_constant = info[info["top_share"] >= 0.99].index.tolist()

# weight is almost all missing, so drop it too
drop_more = near_constant + ["weight"]
print(len(drop_more), drop_more)

# Drop the same columns from train, validation and test (chosen from training only)
X_train = X_train.drop(columns=drop_more)
X_val = X_val.drop(columns=drop_more)
X_test = X_test.drop(columns=drop_more)
print(X_train.shape, X_val.shape, X_test.shape)

16 ['nateglinide', 'chlorpropamide', 'acetohexamide', 'tolbutamide', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'weight']
(69519, 31) (14911, 31) (14913, 31)


In [16]:
# Columns where "?" means missing
q_cols = ["race", "payer_code", "medical_specialty", "diag_1", "diag_2", "diag_3"]

# Columns where NaN means "test was not done"
test_cols = ["max_glu_serum", "A1Cresult"]

# Apply the same fixed words to train, validation and test
for part in [X_train, X_val, X_test]:
    part[q_cols] = part[q_cols].replace("?", "Unknown")
    part[test_cols] = part[test_cols].fillna("NotTested")

In [17]:
# Count remaining "?" and NaN in each part (all should be 0)
for name, part in [("train", X_train), ("val", X_val), ("test", X_test)]:
    print(name, (part == "?").sum().sum(), part.isnull().sum().sum())

# Look at the new categories
print(X_train["max_glu_serum"].value_counts())
print(X_train["race"].value_counts())

train 0 0
val 0 0
test 0 0
max_glu_serum
NotTested    65940
Norm          1806
>200           992
>300           781
Name: count, dtype: int64
race
Caucasian          51880
AfricanAmerican    13177
Unknown             1616
Hispanic            1409
Other               1026
Asian                411
Name: count, dtype: int64


In [18]:
# Turn a diagnosis code into a broad disease group (fixed medical ranges)
def group_diag(code):
    if code == "Unknown":
        return "Unknown"
    if code.startswith(("V", "E")):          # special codes, not diseases
        return "Other"
    n = float(code)
    if 250 <= n < 251:                       return "Diabetes"
    if 390 <= n <= 459 or n == 785:          return "Circulatory"
    if 460 <= n <= 519 or n == 786:          return "Respiratory"
    if 520 <= n <= 579 or n == 787:          return "Digestive"
    if 580 <= n <= 629 or n == 788:          return "Genitourinary"
    if 710 <= n <= 739:                      return "Musculoskeletal"
    if 800 <= n <= 999:                      return "Injury"
    if 140 <= n <= 239:                      return "Neoplasms"
    return "Other"

# Apply the same rule to train, validation and test
for part in [X_train, X_val, X_test]:
    for c in ["diag_1", "diag_2", "diag_3"]:
        part[c] = part[c].map(group_diag)

In [19]:
# Size of each group in the training set, for all three diagnosis columns
print(X_train[["diag_1", "diag_2", "diag_3"]].apply(lambda c: c.value_counts()))

                 diag_1  diag_2  diag_3
Circulatory       20822   21787   20795
Diabetes           6081    8891   11869
Digestive          6549    2843    2690
Genitourinary      3435    5694    4482
Injury             4758    1659    1322
Musculoskeletal    3485    1258    1335
Neoplasms          2181    1628    1137
Other             12386   18154   19933
Respiratory        9808    7344    4926
Unknown              14     261    1030


In [20]:
# How often each code appears in training (count and share)
for c in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
    counts = X_train[c].value_counts()
    print(c)
    print(pd.DataFrame({"count": counts, "share": (counts / len(X_train)).round(3)}))
    print()

admission_type_id
                   count  share
admission_type_id              
1                  36577  0.526
3                  13026  0.187
2                  12785  0.184
6                   3657  0.053
5                   3237  0.047
8                    216  0.003
7                     13  0.000
4                      8  0.000

discharge_disposition_id
                          count  share
discharge_disposition_id              
1                         42204  0.607
3                          9756  0.140
6                          9019  0.130
18                         2596  0.037
2                          1488  0.021
22                         1401  0.020
5                           806  0.012
25                          682  0.010
4                           579  0.008
7                           408  0.006
23                          291  0.004
28                           88  0.001
8                            77  0.001
15                           48  0.001
24          

In [21]:
# Build a lookup: each code -> its group name. Any code not listed becomes "Other"
def make_map(groups):
    lookup = {}
    for group, codes in groups.items():
        for c in codes:
            lookup[c] = group
    return lookup

admission_type_map = make_map({
    "Emergency": [1], "Urgent": [2], "Elective": [3],
    "Unknown": [5, 6, 8],            # Not Available, NULL, Not Mapped
})
discharge_map = make_map({
    "Home": [1], "HomeHealth": [6, 8], "LeftAMA": [7],
    "Facility": [2, 3, 4, 5, 22, 23, 24, 29, 30],   # transferred to another care facility
    "Unknown": [18, 25, 26],         # NULL, Not Mapped, Unknown/Invalid
})
source_map = make_map({
    "Emergency": [7], "Referral": [1, 2, 3],
    "Transfer": [4, 5, 6, 10, 18, 22, 25, 26],
    "Unknown": [9, 15, 17, 20, 21],  # Not Available, NULL, Not Mapped, Unknown
})

# Apply the same grouping to train, validation and test
for part in [X_train, X_val, X_test]:
    part["admission_type_id"] = part["admission_type_id"].map(admission_type_map).fillna("Other")
    part["discharge_disposition_id"] = part["discharge_disposition_id"].map(discharge_map).fillna("Other")
    part["admission_source_id"] = part["admission_source_id"].map(source_map).fillna("Other")

In [22]:
# Size of each group in the training set
for c in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
    print(X_train[c].value_counts(), "\n")

admission_type_id
Emergency    36577
Elective     13026
Urgent       12785
Unknown       7110
Other           21
Name: count, dtype: int64 

discharge_disposition_id
Home          42204
Facility      14352
HomeHealth     9096
Unknown        3278
LeftAMA         408
Other           181
Name: count, dtype: int64 

admission_source_id
Emergency    39025
Referral     21329
Unknown       4786
Transfer      4366
Other           13
Name: count, dtype: int64 



In [23]:
# How many training patients are in each age group, in age order
print(X_train["age"].value_counts().sort_index())

age
[0-10)        118
[10-20)       484
[20-30)      1089
[30-40)      2668
[40-50)      6718
[50-60)     11748
[60-70)     15628
[70-80)     17777
[80-90)     11519
[90-100)     1770
Name: count, dtype: int64


In [24]:
# Map each age group to its middle age (for example [50-60) becomes 55)
age_map = {"[0-10)": 5, "[10-20)": 15, "[20-30)": 25, "[30-40)": 35, "[40-50)": 45,
           "[50-60)": 55, "[60-70)": 65, "[70-80)": 75, "[80-90)": 85, "[90-100)": 95}

# Apply the same map to train, validation and test
for part in [X_train, X_val, X_test]:
    part["age"] = part["age"].map(age_map)

# Check: no missing values, and the numbers look right
print(X_train["age"].isnull().sum(), X_val["age"].isnull().sum(), X_test["age"].isnull().sum())
print(X_train["age"].describe().round(1))

0 0 0
count    69519.0
mean        65.8
std         15.9
min          5.0
25%         55.0
50%         65.0
75%         75.0
max         95.0
Name: age, dtype: float64


In [25]:
# The 8 medication columns still in the data
med_cols = ["metformin", "repaglinide", "glimepiride", "glipizide",
            "glyburide", "pioglitazone", "rosiglitazone", "insulin"]

# Count of each value in the training set
print(X_train[med_cols].apply(lambda c: c.value_counts()).fillna(0).astype(int))

# Also look at the two yes/no columns about medication
print(X_train["change"].value_counts())
print(X_train["diabetesMed"].value_counts())

        metformin  repaglinide  glimepiride  glipizide  glyburide  \
Down          407           29          148        370        411   
No          55451        68439        65818      60778      62168   
Steady      12913          966         3329       7823       6371   
Up            748           85          224        548        569   

        pioglitazone  rosiglitazone  insulin  
Down              86             60     8283  
No             64432          65037    32311  
Steady          4844           4300    21225  
Up               157            122     7700  
change
No    37146
Ch    32373
Name: count, dtype: int64
diabetesMed
Yes    53867
No     15652
Name: count, dtype: int64


In [26]:
# The 7 drugs where Up and Down are rare (insulin is kept as it is)
rare_meds = ["metformin", "repaglinide", "glimepiride", "glipizide",
             "glyburide", "pioglitazone", "rosiglitazone"]

for part in [X_train, X_val, X_test]:
    # Up and Down both mean the dose was changed
    part[rare_meds] = part[rare_meds].replace({"Up": "Changed", "Down": "Changed"})
    # Yes/no columns become 1 and 0
    part["change"] = (part["change"] == "Ch").astype(int)
    part["diabetesMed"] = (part["diabetesMed"] == "Yes").astype(int)

In [27]:
# Values of two of the changed drug columns, plus insulin
print(X_train[["metformin", "repaglinide", "insulin"]].apply(lambda c: c.value_counts()).fillna(0).astype(int))

# The two yes/no columns should now be 0 and 1
print(X_train["change"].value_counts())
print(X_train["diabetesMed"].value_counts())

         metformin  repaglinide  insulin
Changed       1155          114        0
Down             0            0     8283
No           55451        68439    32311
Steady       12913          966    21225
Up               0            0     7700
change
0    37146
1    32373
Name: count, dtype: int64
diabetesMed
1    53867
0    15652
Name: count, dtype: int64


In [28]:
# Show the 15 most common values and how many values are rare (under 100 patients)
for c in ["medical_specialty", "payer_code", "gender"]:
    counts = X_train[c].value_counts()
    print(c, "| values:", len(counts), "| rare (under 100):", (counts < 100).sum())
    print(counts.head(15), "\n")

medical_specialty | values: 72 | rare (under 100): 48
medical_specialty
Unknown                            33896
InternalMedicine                   10093
Emergency/Trauma                    5213
Family/GeneralPractice              5052
Cardiology                          3752
Surgery-General                     2137
Nephrology                          1056
Orthopedics                          947
Orthopedics-Reconstructive           853
Radiologist                          772
Pulmonology                          617
Psychiatry                           597
Urology                              487
Surgery-Cardiovascular/Thoracic      452
ObstetricsandGynecology              449
Name: count, dtype: int64 

payer_code | values: 18 | rare (under 100): 6
payer_code
Unknown    27691
MC         21919
HM          4278
SP          3387
BC          3221
MD          2437
CP          1743
UN          1675
CM          1322
OG           698
DM           401
PO           398
WC            94
CH     

In [29]:
# Keep a value only if it has at least 500 patients in the TRAINING set
min_count = 500
keep = {}
for c in ["medical_specialty", "payer_code"]:
    counts = X_train[c].value_counts()
    keep[c] = counts[counts >= min_count].index.tolist()

# gender: the most common training value, used to replace "Unknown/Invalid"
gender_mode = X_train["gender"].mode()[0]

# Apply the same rules to train, validation and test
for part in [X_train, X_val, X_test]:
    for c in keep:
        part[c] = part[c].where(part[c].isin(keep[c]), "Other")
    part["gender"] = part["gender"].replace("Unknown/Invalid", gender_mode)

In [30]:
# Number of values and the full list for each column (training set)
for c in ["medical_specialty", "payer_code", "gender"]:
    print(c, "| values:", X_train[c].nunique())
    print(X_train[c].value_counts(), "\n")

medical_specialty | values: 13
medical_specialty
Unknown                       33896
InternalMedicine              10093
Emergency/Trauma               5213
Family/GeneralPractice         5052
Other                          4534
Cardiology                     3752
Surgery-General                2137
Nephrology                     1056
Orthopedics                     947
Orthopedics-Reconstructive      853
Radiologist                     772
Pulmonology                     617
Psychiatry                      597
Name: count, dtype: int64 

payer_code | values: 11
payer_code
Unknown    27691
MC         21919
HM          4278
SP          3387
BC          3221
MD          2437
CP          1743
UN          1675
CM          1322
Other       1148
OG           698
Name: count, dtype: int64 

gender | values: 2
gender
Female    37303
Male      32216
Name: count, dtype: int64 



In [31]:
# The number columns
num_cols = ["age", "time_in_hospital", "num_lab_procedures", "num_procedures",
            "num_medications", "number_outpatient", "number_emergency",
            "number_inpatient", "number_diagnoses"]

# Mean, median, spread and the extremes
summary = X_train[num_cols].agg(["mean", "median", "std", "min", "max"]).T

# IQR rule, the same as in the heart data: count values outside the box limits
Q1, Q3 = X_train[num_cols].quantile(0.25), X_train[num_cols].quantile(0.75)
IQR = Q3 - Q1
summary["outliers"] = ((X_train[num_cols] < Q1 - 1.5 * IQR) | (X_train[num_cols] > Q3 + 1.5 * IQR)).sum()

print(summary.round(2))

                     mean  median    std  min    max  outliers
age                 65.78    65.0  15.90  5.0   95.0       602
time_in_hospital     4.38     4.0   2.97  1.0   14.0      1483
num_lab_procedures  42.98    44.0  19.53  1.0  132.0        74
num_procedures       1.33     1.0   1.70  0.0    6.0      3340
num_medications     15.99    15.0   8.12  1.0   81.0      1735
number_outpatient    0.36     0.0   1.24  0.0   40.0     11417
number_emergency     0.19     0.0   0.94  0.0   76.0      7707
number_inpatient     0.63     0.0   1.25  0.0   21.0      4783
number_diagnoses     7.40     8.0   1.95  1.0   16.0       208


In [32]:
# For the three visit columns: share of zeros, and the values at the top
for c in ["number_outpatient", "number_emergency", "number_inpatient"]:
    s = X_train[c]
    print(c, "| zeros:", round((s == 0).mean(), 3),
          "| 90%:", s.quantile(0.90), "| 99%:", s.quantile(0.99), "| max:", s.max())

number_outpatient | zeros: 0.836 | 90%: 1.0 | 99%: 5.0 | max: 40
number_emergency | zeros: 0.889 | 90%: 1.0 | 99%: 3.0 | max: 76
number_inpatient | zeros: 0.667 | 90%: 2.0 | 99%: 6.0 | max: 21


In [33]:
import numpy as np

# The three visit-count columns with a long tail
visit_cols = ["number_outpatient", "number_emergency", "number_inpatient"]

# log(1 + x) on train, validation and test (run this cell only once)
for part in [X_train, X_val, X_test]:
    part[visit_cols] = np.log1p(part[visit_cols])

In [34]:
# Max and mean after the transform, and the share of zeros should stay the same
print(X_train[visit_cols].agg(["mean", "max"]).round(2))
print((X_train[visit_cols] == 0).mean().round(3))

      number_outpatient  number_emergency  number_inpatient
mean               0.17              0.10              0.32
max                3.71              4.34              3.09
number_outpatient    0.836
number_emergency     0.889
number_inpatient     0.667
dtype: float64


In [35]:
# All the text (category) columns, now small groups
cat_cols = ["race", "gender", "payer_code", "medical_specialty",
            "diag_1", "diag_2", "diag_3",
            "admission_type_id", "discharge_disposition_id", "admission_source_id",
            "max_glu_serum", "A1Cresult",
            "metformin", "repaglinide", "glimepiride", "glipizide",
            "glyburide", "pioglitazone", "rosiglitazone", "insulin"]

# One-hot encode training, then force validation and test to have the same columns
X_train_enc = pd.get_dummies(X_train, columns=cat_cols, dtype=int)
X_val_enc = pd.get_dummies(X_val, columns=cat_cols, dtype=int).reindex(columns=X_train_enc.columns, fill_value=0)
X_test_enc = pd.get_dummies(X_test, columns=cat_cols, dtype=int).reindex(columns=X_train_enc.columns, fill_value=0)

print(X_train_enc.shape, X_val_enc.shape, X_test_enc.shape)

(69519, 122) (14911, 122) (14913, 122)


In [36]:
# Any column that is still text? (should print an empty list)
print(X_train_enc.select_dtypes(include="object").columns.tolist())

# Any missing values? (should print 0)
print(X_train_enc.isnull().sum().sum())

[]
0


In [37]:
from sklearn.preprocessing import StandardScaler

# The 9 real number columns
scale_cols = ["age", "time_in_hospital", "num_lab_procedures", "num_procedures",
              "num_medications", "number_outpatient", "number_emergency",
              "number_inpatient", "number_diagnoses"]

# Copies, so the unscaled version stays available for tree models
X_train_s = X_train_enc.copy()
X_val_s = X_val_enc.copy()
X_test_s = X_test_enc.copy()

# Learn mean and standard deviation from the TRAINING set only
scaler = StandardScaler()
X_train_s[scale_cols] = scaler.fit_transform(X_train_enc[scale_cols])

# Apply the same training mean and std to validation and test
X_val_s[scale_cols] = scaler.transform(X_val_enc[scale_cols])
X_test_s[scale_cols] = scaler.transform(X_test_enc[scale_cols])

In [38]:
# After scaling, the training mean should be about 0 and the std about 1
print(X_train_s[scale_cols].agg(["mean", "std"]).round(2))

# Validation and test should be close, but not exactly 0 and 1
print(X_val_s[scale_cols].agg(["mean", "std"]).round(2))

      age  time_in_hospital  num_lab_procedures  num_procedures  \
mean -0.0              -0.0                 0.0             0.0   
std   1.0               1.0                 1.0             1.0   

      num_medications  number_outpatient  number_emergency  number_inpatient  \
mean             -0.0               -0.0               0.0               0.0   
std               1.0                1.0               1.0               1.0   

      number_diagnoses  
mean               0.0  
std                1.0  
       age  time_in_hospital  num_lab_procedures  num_procedures  \
mean -0.02              0.01               -0.01            0.01   
std   1.01              1.01                1.02            1.01   

      num_medications  number_outpatient  number_emergency  number_inpatient  \
mean             -0.0               0.01              0.00              0.00   
std               1.0               1.03              1.02              1.01   

      number_diagnoses  
mean       

In [39]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

# Baseline: always predict "not readmitted"
base = DummyClassifier(strategy="most_frequent").fit(X_train_s, y_train)
pred = base.predict(X_val_s)

print("accuracy:", round(accuracy_score(y_val, pred), 3))
print("recall:  ", round(recall_score(y_val, pred), 3))
print("AUC:     ", round(roc_auc_score(y_val, base.predict_proba(X_val_s)[:, 1]), 3))

accuracy: 0.883
recall:   0.0
AUC:      0.5


In [40]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, average_precision_score)

# Baseline: always predict the most common class (not readmitted)
base = DummyClassifier(strategy="most_frequent").fit(X_train_s, y_train)
pred = base.predict(X_val_s)

# The mistakes, and the scores
print(confusion_matrix(y_val, pred))
print("accuracy: ", round(accuracy_score(y_val, pred), 3))
print("precision:", round(precision_score(y_val, pred, zero_division=0), 3))
print("recall:   ", round(recall_score(y_val, pred), 3))
print("f1:       ", round(f1_score(y_val, pred), 3))

# The score that "guessing" gets for ranking: equal to the share of positives
print("avg precision (baseline):", round(average_precision_score(y_val, base.predict_proba(X_val_s)[:, 1]), 3))

[[13171     0]
 [ 1740     0]]
accuracy:  0.883
precision: 0.0
recall:    0.0
f1:        0.0
avg precision (baseline): 0.117


In [41]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score

# Each model with the data version it needs
candidates = {
    "Logistic Regression": (LogisticRegression(max_iter=1000, class_weight="balanced"), X_train_s, X_val_s),
    "Random Forest": (RandomForestClassifier(n_estimators=200, min_samples_leaf=20,
                      class_weight="balanced_subsample", n_jobs=-1, random_state=42), X_train_enc, X_val_enc),
    "Gradient Boosting": (HistGradientBoostingClassifier(class_weight="balanced", random_state=42), X_train_enc, X_val_enc),
    "Naive Bayes": (GaussianNB(), X_train_s, X_val_s),
}

rows, matrices, val_scores = [], {}, {}
for name, (model, Xtr, Xva) in candidates.items():
    model.fit(Xtr, y_train)                              # train on the training set
    probs = model.predict_proba(Xva)[:, 1]               # risk score for each validation patient
    val_scores[name] = probs
    pred = (probs >= np.quantile(probs, 0.90)).astype(int)   # flag the top 10% by risk
    matrices[name] = confusion_matrix(y_val, pred)
    rows.append({"model": name,
                 "ROC-AUC": roc_auc_score(y_val, probs),
                 "avg precision": average_precision_score(y_val, probs),
                 "accuracy": accuracy_score(y_val, pred),
                 "precision": precision_score(y_val, pred),
                 "recall": recall_score(y_val, pred),
                 "f1": f1_score(y_val, pred)})

# Summary table, then the confusion matrix of each model
print(pd.DataFrame(rows).set_index("model").round(3))
for name, m in matrices.items():
    print("\n", name); print(m)

                     ROC-AUC  avg precision  accuracy  precision  recall  \
model                                                                      
Logistic Regression    0.664          0.219     0.834      0.252   0.216   
Random Forest          0.672          0.224     0.837      0.269   0.231   
Gradient Boosting      0.667          0.226     0.837      0.270   0.232   
Naive Bayes            0.620          0.176     0.822      0.194   0.166   

                        f1  
model                       
Logistic Regression  0.233  
Random Forest        0.249  
Gradient Boosting    0.249  
Naive Bayes          0.179  

 Logistic Regression
[[12055  1116]
 [ 1364   376]]

 Random Forest
[[12081  1090]
 [ 1338   402]]

 Gradient Boosting
[[12082  1089]
 [ 1337   403]]

 Naive Bayes
[[11968  1203]
 [ 1451   289]]


In [42]:
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Logistic Regression with scaling inside the pipeline, so each fold learns its own scaling
lr_pipe = Pipeline([
    ("prep", ColumnTransformer([("scale", StandardScaler(), scale_cols)], remainder="passthrough")),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
gb = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)

# 5 folds, and all stays of one patient always land in the same fold
cv = GroupKFold(n_splits=5)

for name, model in [("Logistic Regression", lr_pipe), ("Gradient Boosting", gb)]:
    res = cross_validate(model, X_train_enc, y_train, groups=g_train, cv=cv,
                         scoring=["roc_auc", "average_precision"], n_jobs=-1)
    for m in ["roc_auc", "average_precision"]:
        s = res["test_" + m]
        print(f"{name} | {m}: {s.mean():.3f} (+/- {s.std():.3f})")

Logistic Regression | roc_auc: 0.657 (+/- 0.005)
Logistic Regression | average_precision: 0.200 (+/- 0.011)
Gradient Boosting | roc_auc: 0.664 (+/- 0.005)
Gradient Boosting | average_precision: 0.212 (+/- 0.008)


In [43]:
from sklearn.model_selection import RandomizedSearchCV

# The settings to try
param_grid = {
    "learning_rate": [0.03, 0.05, 0.1],
    "max_leaf_nodes": [8, 15, 31],
    "min_samples_leaf": [20, 50, 100, 200],
    "l2_regularization": [0, 1, 10],
    "max_iter": [200, 400],
}

# Try 12 random combinations, each scored with 5-fold cross-validation by patient
search = RandomizedSearchCV(
    HistGradientBoostingClassifier(class_weight="balanced", early_stopping=True, random_state=42),
    param_distributions=param_grid, n_iter=12, scoring="average_precision",
    cv=GroupKFold(n_splits=5), n_jobs=-1, random_state=42)
search.fit(X_train_enc, y_train, groups=g_train)

# The best settings, and the top 5 combinations
print("best:", search.best_params_)
print("best avg precision:", round(search.best_score_, 3))
res = pd.DataFrame(search.cv_results_)
print(res.sort_values("rank_test_score")[["mean_test_score", "std_test_score", "params"]].head(5).round(3))

best: {'min_samples_leaf': 50, 'max_leaf_nodes': 31, 'max_iter': 200, 'learning_rate': 0.05, 'l2_regularization': 10}
best avg precision: 0.216
    mean_test_score  std_test_score  \
3             0.216           0.007   
0             0.215           0.007   
1             0.215           0.007   
8             0.214           0.009   
11            0.214           0.009   

                                               params  
3   {'min_samples_leaf': 50, 'max_leaf_nodes': 31,...  
0   {'min_samples_leaf': 50, 'max_leaf_nodes': 31,...  
1   {'min_samples_leaf': 50, 'max_leaf_nodes': 31,...  
8   {'min_samples_leaf': 50, 'max_leaf_nodes': 8, ...  
11  {'min_samples_leaf': 100, 'max_leaf_nodes': 15...  


In [44]:
from sklearn.model_selection import GridSearchCV

# Same pipeline as before: scale inside each fold, then Logistic Regression
lr_pipe = Pipeline([
    ("prep", ColumnTransformer([("scale", StandardScaler(), scale_cols)], remainder="passthrough")),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

# Try 5 values of C (smaller C = stronger penalty)
lr_search = GridSearchCV(lr_pipe, {"model__C": [0.001, 0.01, 0.1, 1, 10]},
                         scoring="average_precision", cv=GroupKFold(n_splits=5), n_jobs=-1)
lr_search.fit(X_train_enc, y_train, groups=g_train)

print("best C:", lr_search.best_params_)
print("best avg precision:", round(lr_search.best_score_, 3))
print(pd.DataFrame(lr_search.cv_results_)[["param_model__C", "mean_test_score", "std_test_score"]].round(3))

best C: {'model__C': 0.001}
best avg precision: 0.2
   param_model__C  mean_test_score  std_test_score
0           0.001              0.2           0.010
1           0.010              0.2           0.011
2           0.100              0.2           0.011
3           1.000              0.2           0.011
4          10.000              0.2           0.011


In [45]:
# The tuned models (both already trained on the full training set)
tuned = {"Logistic Regression": lr_search.best_estimator_,
         "Gradient Boosting": search.best_estimator_}

for name, m in tuned.items():
    probs = m.predict_proba(X_val_enc)[:, 1]
    print(name, "| ROC-AUC", round(roc_auc_score(y_val, probs), 3),
          "| avg precision", round(average_precision_score(y_val, probs), 3))

    # Flag the top 5%, 10%, 20%, 30% by risk and measure what we catch
    for share in [0.05, 0.10, 0.20, 0.30]:
        flagged = probs >= np.quantile(probs, 1 - share)
        caught = (flagged & (y_val.values == 1)).sum()
        print(f"  top {int(share*100)}%: recall {caught/y_val.sum():.2f} | "
              f"precision {caught/flagged.sum():.2f} | lift {caught/flagged.sum()/y_val.mean():.1f}x")

    # Confusion matrix when flagging the top 10%
    pred = (probs >= np.quantile(probs, 0.90)).astype(int)
    print(confusion_matrix(y_val, pred), "\n")

Logistic Regression | ROC-AUC 0.66 | avg precision 0.209
  top 5%: recall 0.13 | precision 0.29 | lift 2.5x
  top 10%: recall 0.21 | precision 0.24 | lift 2.1x
  top 20%: recall 0.37 | precision 0.22 | lift 1.8x
  top 30%: recall 0.50 | precision 0.19 | lift 1.7x
[[12043  1128]
 [ 1376   364]] 

Gradient Boosting | ROC-AUC 0.67 | avg precision 0.23
  top 5%: recall 0.14 | precision 0.33 | lift 2.8x
  top 10%: recall 0.24 | precision 0.28 | lift 2.4x
  top 20%: recall 0.39 | precision 0.22 | lift 1.9x
  top 30%: recall 0.51 | precision 0.20 | lift 1.7x
[[12100  1071]
 [ 1319   421]] 



In [46]:
# Final model: tuned Gradient Boosting (already trained on the full training set)
final = search.best_estimator_

# Flag size chosen by the hospital (10% as the example). The cutoff comes from VALIDATION.
share = 0.10
val_probs = final.predict_proba(X_val_enc)[:, 1]
cutoff = np.quantile(val_probs, 1 - share)
print("cutoff (from validation):", round(cutoff, 3))

# Apply that fixed cutoff to the test set, once
test_probs = final.predict_proba(X_test_enc)[:, 1]
flagged = test_probs >= cutoff
caught = (flagged & (y_test.values == 1)).sum()

print("flagged:", f"{flagged.mean():.1%}", "of patients")
print("recall:", round(caught / y_test.sum(), 2))
print("precision:", round(caught / flagged.sum(), 2))
print("lift:", round((caught / flagged.sum()) / y_test.mean(), 1), "x")
print("ROC-AUC:", round(roc_auc_score(y_test, test_probs), 3))
print("avg precision:", round(average_precision_score(y_test, test_probs), 3))
print(confusion_matrix(y_test, flagged.astype(int)))

cutoff (from validation): 0.641
flagged: 9.9% of patients
recall: 0.22
precision: 0.25
lift: 2.2 x
ROC-AUC: 0.671
avg precision: 0.211
[[12144  1111]
 [ 1288   370]]
